In [0]:
import os
import time
import base64
import requests
import jwt

In [0]:
# GitHub repo target
repo_owner = "KingBain"
repo_name = "fsdh-artemis-demo"
branch_name = "main"

# Databricks secret scope
secret_scope = "github-app"

# Local generated site root
site_root = "/Workspace/Users/john.bain@ssc-spc.gc.ca/Artemis 2/fsdh-artemis-demo"

# Files to publish: local path -> repo path
files_to_publish = {
    os.path.join(site_root, "index.html"): "index.html",
    os.path.join(site_root, "assets", "orion-speed-over-time.png"): "assets/orion-speed-over-time.png",
    os.path.join(site_root, "assets", "orion-distance-over-time.png"): "assets/orion-distance-over-time.png",
    os.path.join(site_root, "assets", "orion-summary-card.png"): "assets/orion-summary-card.png",
}

github_api_base = "https://api.github.com"

In [0]:
app_id = dbutils.secrets.get(scope=secret_scope, key="app-id")
private_key_pem = dbutils.secrets.get(scope=secret_scope, key="private-key")

print("GitHub App secrets loaded successfully.")
print(f"App ID length: {len(app_id)}")
print(f"Private key length: {len(private_key_pem)}")

In [0]:
now = int(time.time())

jwt_payload = {
    "iat": now - 60,
    "exp": now + (9 * 60),
    "iss": app_id,
}

app_jwt = jwt.encode(jwt_payload, private_key_pem, algorithm="RS256")

print("Generated GitHub App JWT.")

In [0]:
app_headers = {
    "Authorization": f"Bearer {app_jwt}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

installation_url = f"{github_api_base}/repos/{repo_owner}/{repo_name}/installation"
installation_resp = requests.get(installation_url, headers=app_headers, timeout=30)

if installation_resp.status_code != 200:
    raise Exception(
        f"Failed to resolve installation for repo: "
        f"{installation_resp.status_code} {installation_resp.text}"
    )

installation_id = installation_resp.json()["id"]
print(f"Resolved installation ID: {installation_id}")

In [0]:
token_url = f"{github_api_base}/app/installations/{installation_id}/access_tokens"
token_resp = requests.post(token_url, headers=app_headers, timeout=30)

if token_resp.status_code not in (200, 201):
    raise Exception(
        f"Failed to create installation token: "
        f"{token_resp.status_code} {token_resp.text}"
    )

token_data = token_resp.json()
installation_token = token_data["token"]
expires_at = token_data["expires_at"]

print(f"Generated installation token. Expires at: {expires_at}")

In [0]:
github_headers = {
    "Authorization": f"Bearer {installation_token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

In [0]:
def read_file_base64(local_path: str) -> str:
    with open(local_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def get_existing_file_sha(repo_path: str):
    url = f"{github_api_base}/repos/{repo_owner}/{repo_name}/contents/{repo_path}"
    resp = requests.get(
        url,
        headers=github_headers,
        params={"ref": branch_name},
        timeout=30,
    )

    if resp.status_code == 200:
        return resp.json()["sha"]

    if resp.status_code == 404:
        return None

    raise Exception(
        f"Failed checking existing file {repo_path}: "
        f"{resp.status_code} {resp.text}"
    )


def create_or_update_file(local_path: str, repo_path: str, commit_message: str):
    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Local file does not exist: {local_path}")

    content_b64 = read_file_base64(local_path)
    existing_sha = get_existing_file_sha(repo_path)

    payload = {
        "message": commit_message,
        "content": content_b64,
        "branch": branch_name,
    }

    if existing_sha:
        payload["sha"] = existing_sha

    url = f"{github_api_base}/repos/{repo_owner}/{repo_name}/contents/{repo_path}"
    resp = requests.put(url, headers=github_headers, json=payload, timeout=60)

    if resp.status_code not in (200, 201):
        raise Exception(
            f"Failed publishing {repo_path}: "
            f"{resp.status_code} {resp.text}"
        )

    return resp.json()

In [0]:
print("Checking local files to publish...")
missing_files = []

for local_path, repo_path in files_to_publish.items():
    exists = os.path.exists(local_path)
    print(f"- {local_path} -> {repo_path} | exists={exists}")
    if not exists:
        missing_files.append(local_path)

if missing_files:
    raise Exception(f"Cannot publish because files are missing: {missing_files}")

In [0]:
results = []

for local_path, repo_path in files_to_publish.items():
    print(f"Publishing {repo_path} ...")

    result = create_or_update_file(
        local_path=local_path,
        repo_path=repo_path,
        commit_message=f"Update Artemis II site asset: {repo_path}"
    )

    results.append({
        "repo_path": repo_path,
        "commit_sha": result["commit"]["sha"],
    })

    print(f"Published {repo_path}")

In [0]:
print("Publish complete.")
for item in results:
    print(f"- {item['repo_path']} | commit {item['commit_sha']}")